# Consult-Tech - Workforce Analytics & Inferential Modelling

**Track 2 (Product & Consulting) - Analytics Foundation Notebook.**

Consult-Tech is advising a client on opening a new R&D centre in **Bengaluru, India** - a high-inflation region - while the client manages elevated employee attrition. This notebook produces the analytics that feed the two final deliverables:

1. An **Excel `.xlsx` Market Feasibility Model** with Scenario Manager (built directly in Python via openpyxl, opens natively in Excel)
2. A **Power BI dashboard** built from a documented `.pbix` spec (Power Query M code, DAX measures, visual layout)

**What this notebook does:**
- Cleans the HR data using the same logic that goes into Power Query M
- Pre-computes every DAX measure so you know what each card on the dashboard should show
- Runs the Central Limit Theorem analysis with bootstrap simulations and confidence intervals
- Generates the input numbers feeding the Excel scenario model

**Dataset:** IBM HR Analytics (1,470 employees, 35 cols) + India macro indicators 2015-2025.

## Setup - auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 02/
#     └── Track 02/
#         ├── dataset/                          (HR CSV + macro CSV here)
#         └── consult_tech/
#             ├── data/
#             ├── outputs/
#             ├── figures/
#             └── (this notebook)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "consult_tech":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate input files
HR_FILE = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
MACRO_FILE = "india_macro_indicators.csv"
file_paths = {}
for name in [HR_FILE, MACRO_FILE]:
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            file_paths[name] = folder / name
            break
missing = [f for f in [HR_FILE, MACRO_FILE] if f not in file_paths]
assert not missing, f"Missing files: {missing}. Place them in {DATASET_DIR} or {DATA_DIR}"
print(f"Found {len(file_paths)} files: {list(file_paths.keys())}")


## Part 1 - Power Query equivalent cleanup

What we do here in pandas mirrors exactly what Power Query M will do in Power BI. Each step is documented so the M code in the dashboard spec produces the identical cleaned table.

### Step 1.1 - Load raw data and audit

In [ ]:
import pandas as pd
import numpy as np

hr_raw = pd.read_csv(file_paths["WA_Fn-UseC_-HR-Employee-Attrition.csv"])
macro = pd.read_csv(file_paths["india_macro_indicators.csv"])

print(f"HR data: {hr_raw.shape}")
print(f"  Columns: {len(hr_raw.columns)}")
print(f"  Nulls per column: {hr_raw.isna().sum().sum()} total")
print(f"  Duplicate EmployeeNumbers: {hr_raw.duplicated('EmployeeNumber').sum()}")
print(f"\nMacro data: {macro.shape}")
print(macro.to_string(index=False))


### Step 1.2 - Drop degenerate columns

Three columns have only one unique value across all 1,470 rows. They carry no information and clutter the model. The Power BI Power Query equivalent is `Table.RemoveColumns`.

In [ ]:
degenerate_cols = [c for c in hr_raw.columns if hr_raw[c].nunique() == 1]
print(f"Degenerate columns (single value, will drop): {degenerate_cols}")
for c in degenerate_cols:
    print(f"  {c}: only value = {hr_raw[c].iloc[0]}")

hr = hr_raw.drop(columns=degenerate_cols).copy()
print(f"\nAfter cleanup: {hr.shape}")


### Step 1.3 - Add derived columns

These mirror what we'll build in Power Query M (using `Table.AddColumn`):
- `AttritionFlag` (1/0): for sums and averages in DAX
- `TenureBucket`: categorical for the dashboard drill-down
- `IncomeBand`: salary bracket for the supply/demand analysis
- `OverTimeFlag` (1/0)
- `IsHighRisk`: heuristic flag for the executive summary

In [ ]:
hr["AttritionFlag"] = (hr["Attrition"] == "Yes").astype(int)
hr["OverTimeFlag"]  = (hr["OverTime"] == "Yes").astype(int)

hr["TenureBucket"] = pd.cut(
    hr["YearsAtCompany"],
    bins=[-0.1, 1, 3, 5, 10, 100],
    labels=["0-1y", "2-3y", "4-5y", "6-10y", "11+y"],
)

hr["IncomeBand"] = pd.cut(
    hr["MonthlyIncome"],
    bins=[0, 3000, 6000, 10000, 100000],
    labels=["Junior (<3k)", "Mid (3-6k)", "Senior (6-10k)", "Lead (10k+)"],
)

# High-risk = OverTime + low tenure + below-median income (heuristic for the dashboard cards)
median_inc = hr["MonthlyIncome"].median()
hr["IsHighRisk"] = (
    (hr["OverTimeFlag"] == 1) &
    (hr["YearsAtCompany"] <= 3) &
    (hr["MonthlyIncome"] < median_inc)
).astype(int)

# Save the cleaned data so other deliverables consume the same inputs
cleaned_path = OUTPUTS_DIR / "hr_cleaned.csv"
hr.to_csv(cleaned_path, index=False)
print(f"Cleaned HR data saved: {cleaned_path}")
print(f"  Rows: {len(hr):,}, Cols: {len(hr.columns)}")
print(f"\nDerived columns added:")
for col in ["AttritionFlag", "OverTimeFlag", "TenureBucket", "IncomeBand", "IsHighRisk"]:
    if hr[col].dtype.name == "category":
        print(f"  {col:<16} {hr[col].value_counts().sort_index().to_dict()}")
    else:
        print(f"  {col:<16} sum={hr[col].sum():,}, mean={hr[col].mean():.4f}")


## Part 2 - DAX measure pre-computation

Every DAX measure that goes into the Power BI dashboard is computed here first in Python. This way you know what each card and chart should display, and you can verify Power BI is producing the right numbers when you build the `.pbix` from the spec.

### Measure 1: Total Employees, Total Attrition, Attrition Rate

In [ ]:
total_employees     = len(hr)
total_attrition     = int(hr["AttritionFlag"].sum())
attrition_rate      = hr["AttritionFlag"].mean()
attrition_rate_pct  = attrition_rate * 100

print(f"[Total Employees]         = {total_employees:,}")
print(f"[Total Attrition]         = {total_attrition:,}")
print(f"[Attrition Rate]          = {attrition_rate:.4f}")
print(f"[Attrition Rate %]        = {attrition_rate_pct:.2f}%")
print(f"\n--- DAX equivalents (the formulas you'll paste into Power BI) ---")
print('  Total Employees     = COUNTROWS(\'HR\')')
print('  Total Attrition     = SUM(\'HR\'[AttritionFlag])')
print('  Attrition Rate      = DIVIDE([Total Attrition], [Total Employees])')
print('  Attrition Rate %    = FORMAT([Attrition Rate], "0.00%")')


### Measure 2: Average Tenure (overall + filtered)

In [ ]:
avg_tenure_overall = hr["YearsAtCompany"].mean()
avg_tenure_stayers = hr.loc[hr["AttritionFlag"]==0, "YearsAtCompany"].mean()
avg_tenure_leavers = hr.loc[hr["AttritionFlag"]==1, "YearsAtCompany"].mean()

print(f"[Average Tenure]          = {avg_tenure_overall:.2f} years (overall)")
print(f"[Avg Tenure - Stayers]    = {avg_tenure_stayers:.2f} years")
print(f"[Avg Tenure - Leavers]    = {avg_tenure_leavers:.2f} years")
print(f"  Difference: leavers stay {avg_tenure_stayers - avg_tenure_leavers:.2f} years less on average")

print(f"\n--- DAX ---")
print('  Average Tenure          = AVERAGE(\'HR\'[YearsAtCompany])')
print('  Avg Tenure (Stayers)    = CALCULATE(AVERAGE(\'HR\'[YearsAtCompany]), \'HR\'[Attrition]="No")')
print('  Avg Tenure (Leavers)    = CALCULATE(AVERAGE(\'HR\'[YearsAtCompany]), \'HR\'[Attrition]="Yes")')


### Measure 3: Rolling Attrition Rate by Tenure Cohort

**Important methodology note.** The brief asks for a 'Rolling Attrition Rate'. The IBM dataset has no hire/termination dates - just `YearsAtCompany`. The defensible proxy (used in every IBM-attrition tutorial) is **tenure-cohort attrition**: bucket employees by years at company and compute attrition rate per cohort. This shows whether attrition risk decays with tenure (it does, dramatically).

In [ ]:
cohort_attrition = hr.groupby("TenureBucket", observed=True).agg(
    n_employees    = ("EmployeeNumber", "count"),
    n_attrition    = ("AttritionFlag", "sum"),
    attrition_rate = ("AttritionFlag", "mean"),
).reset_index()
cohort_attrition["attrition_rate_pct"] = (cohort_attrition["attrition_rate"] * 100).round(2)
print("Rolling Attrition Rate by Tenure Cohort:")
print(cohort_attrition.to_string(index=False))

cohort_attrition.to_csv(OUTPUTS_DIR / "cohort_attrition.csv", index=False)

print(f"\n--- DAX ---")
print('  Rolling Attrition Rate (by Tenure Cohort) =')
print('      DIVIDE(')
print('          CALCULATE(SUM(\'HR\'[AttritionFlag])),')
print('          CALCULATE(COUNTROWS(\'HR\'))')
print('      )')
print('  -- displayed in a clustered bar chart with TenureBucket on the axis')


### Measure 4: Attrition by Department (the executive drill-down)

In [ ]:
dept_attrition = hr.groupby("Department").agg(
    n_employees    = ("EmployeeNumber", "count"),
    n_attrition    = ("AttritionFlag", "sum"),
    attrition_rate = ("AttritionFlag", "mean"),
    avg_income     = ("MonthlyIncome", "mean"),
    avg_tenure     = ("YearsAtCompany", "mean"),
    avg_satisfaction = ("JobSatisfaction", "mean"),
).reset_index()
dept_attrition["attrition_rate_pct"] = (dept_attrition["attrition_rate"] * 100).round(2)
dept_attrition = dept_attrition.sort_values("attrition_rate", ascending=False)
print("Attrition by Department:")
print(dept_attrition.to_string(index=False))

dept_attrition.to_csv(OUTPUTS_DIR / "department_attrition.csv", index=False)


### Measure 5: Attrition by JobRole (the deepest drill-down)

In [ ]:
role_attrition = hr.groupby("JobRole").agg(
    n_employees    = ("EmployeeNumber", "count"),
    n_attrition    = ("AttritionFlag", "sum"),
    attrition_rate = ("AttritionFlag", "mean"),
    avg_income     = ("MonthlyIncome", "mean"),
).reset_index()
role_attrition["attrition_rate_pct"] = (role_attrition["attrition_rate"] * 100).round(2)
role_attrition = role_attrition.sort_values("attrition_rate", ascending=False)
print("Attrition by Job Role:")
print(role_attrition.to_string(index=False))

# The headline finding for the dashboard
worst_role = role_attrition.iloc[0]
print(f"\nHeadline: {worst_role['JobRole']} attrition = {worst_role['attrition_rate_pct']}%")
print(f"  ({int(worst_role['n_attrition'])} of {int(worst_role['n_employees'])} left)")

role_attrition.to_csv(OUTPUTS_DIR / "role_attrition.csv", index=False)


### Measure 6: Market Structure proxy

The brief mentions filtering by 'Market Structure' but the dataset has no market label per employee. The defensible proxy is `BusinessTravel` - employees with Frequent travel are exposed to higher labour-market liquidity (more recruiter contact, more competing offers) which is the operational definition of market exposure. Documented in the dashboard spec.

In [ ]:
market_attrition = hr.groupby("BusinessTravel").agg(
    n_employees    = ("EmployeeNumber", "count"),
    n_attrition    = ("AttritionFlag", "sum"),
    attrition_rate = ("AttritionFlag", "mean"),
).reset_index()
market_attrition["attrition_rate_pct"] = (market_attrition["attrition_rate"] * 100).round(2)
print("Market Structure proxy (BusinessTravel):")
print(market_attrition.to_string(index=False))

market_attrition.to_csv(OUTPUTS_DIR / "market_structure_attrition.csv", index=False)


### Measure 7: OverTime risk multiplier (the most actionable finding)

In [ ]:
ot_attrition = hr.groupby("OverTime").agg(
    n_employees    = ("EmployeeNumber", "count"),
    n_attrition    = ("AttritionFlag", "sum"),
    attrition_rate = ("AttritionFlag", "mean"),
).reset_index()
ot_attrition["attrition_rate_pct"] = (ot_attrition["attrition_rate"] * 100).round(2)
print("Attrition by OverTime:")
print(ot_attrition.to_string(index=False))

ot_yes = ot_attrition.loc[ot_attrition["OverTime"]=="Yes", "attrition_rate"].values[0]
ot_no  = ot_attrition.loc[ot_attrition["OverTime"]=="No",  "attrition_rate"].values[0]
print(f"\nOverTime risk multiplier: {ot_yes/ot_no:.2f}x")
print(f"  Employees on overtime are {ot_yes/ot_no:.1f}x more likely to leave.")


## Part 3 - Central Limit Theorem & Confidence Intervals

**Brief task:** *'Apply the Central Limit Theorem to sample employee satisfaction scores and establish Confidence Intervals for the entire organization's morale.'*

We treat `JobSatisfaction` as the morale variable. The 1,470-employee population is the 'true' population; we draw repeated random samples to demonstrate the CLT empirically, then compute confidence intervals at standard alpha levels.

### Step 3.1 - Population statistics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")

morale = hr["JobSatisfaction"].values
pop_mean = morale.mean()
pop_std  = morale.std(ddof=1)
n_pop = len(morale)
print(f"Population (treating all 1,470 employees as 'truth'):")
print(f"  N      = {n_pop:,}")
print(f"  mean   = {pop_mean:.4f}")
print(f"  std    = {pop_std:.4f}")
print(f"  Distribution: {dict(zip(*np.unique(morale, return_counts=True)))}")
print(f"  This is a 4-point Likert scale - clearly NON-normal at the individual level.")


### Step 3.2 - CLT demonstration: sampling distributions for n = 30, 50, 100, 200

We draw 1,000 random samples of each size and plot the distribution of sample means. The CLT predicts these will be approximately normal regardless of the population shape, with standard error = sigma / sqrt(n).

In [ ]:
np.random.seed(42)

clt_results = []
sample_sizes = [30, 50, 100, 200]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, n in zip(axes, sample_sizes):
    sample_means = np.array([
        np.random.choice(morale, size=n, replace=False).mean()
        for _ in range(1000)
    ])
    se_observed = sample_means.std(ddof=1)
    se_theory   = pop_std / np.sqrt(n)
    clt_results.append({
        "sample_size":  n,
        "n_simulations": 1000,
        "mean_of_means": sample_means.mean(),
        "se_observed":   se_observed,
        "se_theoretical": se_theory,
        "se_ratio":      se_observed / se_theory,
    })
    # Plot
    ax.hist(sample_means, bins=30, color="#1F3864", alpha=0.7, edgecolor="white", density=True)
    # Overlay theoretical normal
    x = np.linspace(sample_means.min(), sample_means.max(), 100)
    ax.plot(x, stats.norm.pdf(x, pop_mean, se_theory),
            color="#C00000", linewidth=2, label="Theoretical N(mu, sigma/sqrt(n))")
    ax.axvline(pop_mean, color="#2E7D32", linewidth=2, linestyle="--",
                label=f"Population mu = {pop_mean:.3f}")
    ax.set_title(f"n = {n}\nSE_obs = {se_observed:.4f}, SE_th = {se_theory:.4f}",
                  fontweight="bold")
    ax.set_xlabel("Sample mean")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Central Limit Theorem - Sampling Distribution of Mean JobSatisfaction",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "clt_sampling_distributions.png", dpi=160, bbox_inches="tight")
plt.show()

clt_df = pd.DataFrame(clt_results)
print("\nCLT validation table:")
print(clt_df.to_string(index=False))
clt_df.to_csv(OUTPUTS_DIR / "clt_simulation_results.csv", index=False)


### Step 3.3 - Confidence Intervals from a single sample

In a real consulting engagement, you run *one* employee survey, not 1,000. The CLT lets you build a CI around your single observed sample mean. We demonstrate at n=100 with 90%, 95%, and 99% confidence levels.

In [ ]:
np.random.seed(2026)
n_sample = 100
single_sample = np.random.choice(morale, size=n_sample, replace=False)
sample_mean = single_sample.mean()
sample_std  = single_sample.std(ddof=1)
sample_se   = sample_std / np.sqrt(n_sample)

# Use t-distribution (more conservative; appropriate when sigma estimated from sample)
df_sample = n_sample - 1
print(f"Single sample of n = {n_sample}:")
print(f"  Sample mean = {sample_mean:.4f}")
print(f"  Sample std  = {sample_std:.4f}")
print(f"  Standard Error = {sample_se:.4f}")
print()

ci_results = []
for confidence in [0.90, 0.95, 0.99]:
    alpha = 1 - confidence
    t_crit = stats.t.ppf(1 - alpha/2, df_sample)
    margin = t_crit * sample_se
    ci_low  = sample_mean - margin
    ci_high = sample_mean + margin
    contains_truth = ci_low <= pop_mean <= ci_high
    print(f"  {int(confidence*100)}% CI: [{ci_low:.4f}, {ci_high:.4f}]  "
          f"(margin +/-{margin:.4f}, contains true mu={pop_mean:.4f}: {contains_truth})")
    ci_results.append({
        "confidence_level": confidence,
        "alpha":            alpha,
        "t_critical":       t_crit,
        "sample_mean":      sample_mean,
        "margin_of_error":  margin,
        "ci_low":           ci_low,
        "ci_high":          ci_high,
        "contains_pop_mean": contains_truth,
    })

ci_df = pd.DataFrame(ci_results)
ci_df.to_csv(OUTPUTS_DIR / "confidence_intervals.csv", index=False)


### Step 3.4 - Visualise the confidence intervals

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Plot the population distribution (for context)
ax.axvspan(1, 4, alpha=0.05, color="grey")
ax.axvline(pop_mean, color="#2E7D32", linewidth=2.5,
            label=f"True population mu = {pop_mean:.4f}", zorder=5)
ax.axvline(sample_mean, color="#1F3864", linewidth=2, linestyle="--",
            label=f"Sample mean (n={n_sample}) = {sample_mean:.4f}")

# Plot the three CIs as horizontal error bars
y_levels = [1, 2, 3]
colors = ["#A9D08E", "#E07A1F", "#C00000"]
for i, (level, row) in enumerate(zip([0.90, 0.95, 0.99], ci_results)):
    ax.errorbar(
        row["sample_mean"], y_levels[i],
        xerr=[[row["sample_mean"]-row["ci_low"]], [row["ci_high"]-row["sample_mean"]]],
        fmt="o", markersize=10, color=colors[i], linewidth=3, capsize=8,
        label=f"{int(level*100)}% CI: [{row['ci_low']:.3f}, {row['ci_high']:.3f}]",
    )

ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["90% CI", "95% CI", "99% CI"], fontsize=11)
ax.set_xlabel("Mean JobSatisfaction (4-point Likert)")
ax.set_title("Confidence Intervals for Organization Morale\n"
             f"From a single random sample of n = {n_sample} employees",
              fontweight="bold")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(2.4, 3.1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confidence_intervals_morale.png", dpi=160, bbox_inches="tight")
plt.show()


### Step 3.5 - CIs for all four satisfaction dimensions

Build a full morale snapshot table - the 'morale dashboard' the C-suite wants. All four 1-4 Likert dimensions, each with a 95% CI from a fresh n=100 sample.

In [ ]:
morale_dimensions = ["JobSatisfaction", "EnvironmentSatisfaction",
                      "WorkLifeBalance", "RelationshipSatisfaction", "JobInvolvement"]

morale_summary = []
np.random.seed(7)
for dim in morale_dimensions:
    pop = hr[dim].values
    sample = np.random.choice(pop, size=100, replace=False)
    m  = sample.mean()
    sd = sample.std(ddof=1)
    se = sd / np.sqrt(100)
    t_crit = stats.t.ppf(0.975, 99)
    margin = t_crit * se
    morale_summary.append({
        "dimension":         dim,
        "population_mean":   pop.mean(),
        "sample_mean":       m,
        "sample_std":        sd,
        "se":                se,
        "ci_95_low":         m - margin,
        "ci_95_high":        m + margin,
        "ci_contains_truth": (m - margin) <= pop.mean() <= (m + margin),
    })

morale_df = pd.DataFrame(morale_summary)
morale_df_disp = morale_df.copy()
for c in ["population_mean","sample_mean","sample_std","se","ci_95_low","ci_95_high"]:
    morale_df_disp[c] = morale_df_disp[c].round(4)
print("Morale snapshot - 95% CIs from a single sample of n=100:")
print(morale_df_disp.to_string(index=False))
morale_df.to_csv(OUTPUTS_DIR / "morale_snapshot.csv", index=False)


## Part 4 - Inputs for the Excel Scenario Manager model

The Excel model needs four sets of numbers. We compute them here so the workbook can pre-populate the cells - the Excel file built later will reference these exact values.

### Step 4.1 - Macro indicator data

In [ ]:
print("India macro indicators (2015-2025):")
print(macro.to_string(index=False))

print(f"\nKey statistics for the scenario model:")
print(f"  Mean CPI inflation 2015-2025: {macro['CPI_Inflation_Pct'].mean():.2f}%")
print(f"  Min/Max CPI: {macro['CPI_Inflation_Pct'].min():.2f}% / {macro['CPI_Inflation_Pct'].max():.2f}%")
print(f"  Mean Repo Rate: {macro['RBI_Repo_Rate_EOY_Pct'].mean():.2f}%")
print(f"  Latest year ({macro['Year'].max()}): CPI={macro.iloc[-1]['CPI_Inflation_Pct']}%, Repo={macro.iloc[-1]['RBI_Repo_Rate_EOY_Pct']}%")


### Step 4.2 - Talent supply/demand baseline

We treat the IBM HR data as the 'baseline' GDP-of-talent for the new R&D centre. The Excel model will scale these by inflation/interest rate scenarios.

In [ ]:
# Baseline assumptions for the new R&D centre (200-employee plan)
PLAN_HEADCOUNT = 200
PLAN_RD_SHARE  = 0.65  # 65% R&D, 25% sales support, 10% G&A based on consulting industry norm

# Use IBM HR R&D and Sales averages as baseline costs
baseline_avg_income_rd    = hr.loc[hr["Department"]=="Research & Development", "MonthlyIncome"].mean()
baseline_avg_income_sales = hr.loc[hr["Department"]=="Sales", "MonthlyIncome"].mean()
baseline_avg_income_hr    = hr.loc[hr["Department"]=="Human Resources", "MonthlyIncome"].mean()

# Convert from IBM units to INR equivalent (IBM data is in fictional units; we treat 1 unit = 1 INR per month for the model
# Real consulting firms would replace these with local salary survey data)

baseline_attrition_rd    = hr.loc[hr["Department"]=="Research & Development", "AttritionFlag"].mean()
baseline_attrition_sales = hr.loc[hr["Department"]=="Sales", "AttritionFlag"].mean()
baseline_attrition_hr    = hr.loc[hr["Department"]=="Human Resources", "AttritionFlag"].mean()

# Cost-of-attrition factor: industry standard is ~50-200% of annual salary
# We use 75% as a conservative baseline
COST_OF_REPLACEMENT_PCT = 0.75

print(f"Baseline assumptions (new R&D centre, plan = {PLAN_HEADCOUNT} employees):")
print(f"\n  By Department (baseline from IBM HR):")
print(f"    R&D    : avg monthly income = {baseline_avg_income_rd:>8,.0f}, "
      f"attrition = {baseline_attrition_rd*100:5.2f}%")
print(f"    Sales  : avg monthly income = {baseline_avg_income_sales:>8,.0f}, "
      f"attrition = {baseline_attrition_sales*100:5.2f}%")
print(f"    HR     : avg monthly income = {baseline_avg_income_hr:>8,.0f}, "
      f"attrition = {baseline_attrition_hr*100:5.2f}%")
print(f"\n  Cost-of-replacement factor: {COST_OF_REPLACEMENT_PCT*100:.0f}% of annual salary")

scenario_inputs = pd.DataFrame([{
    "parameter": "plan_headcount",          "value": PLAN_HEADCOUNT,
    "label": "Plan headcount for new R&D centre",
}, {
    "parameter": "rd_headcount_share",      "value": PLAN_RD_SHARE,
    "label": "Share of headcount in R&D",
}, {
    "parameter": "baseline_avg_income_rd",  "value": round(baseline_avg_income_rd),
    "label": "Baseline R&D monthly income (units)",
}, {
    "parameter": "baseline_avg_income_sales", "value": round(baseline_avg_income_sales),
    "label": "Baseline Sales monthly income (units)",
}, {
    "parameter": "baseline_avg_income_hr",   "value": round(baseline_avg_income_hr),
    "label": "Baseline HR monthly income (units)",
}, {
    "parameter": "baseline_attrition_rd",    "value": round(baseline_attrition_rd, 4),
    "label": "Baseline R&D attrition rate",
}, {
    "parameter": "baseline_attrition_sales", "value": round(baseline_attrition_sales, 4),
    "label": "Baseline Sales attrition rate",
}, {
    "parameter": "baseline_attrition_hr",    "value": round(baseline_attrition_hr, 4),
    "label": "Baseline HR attrition rate",
}, {
    "parameter": "cost_of_replacement_pct",  "value": COST_OF_REPLACEMENT_PCT,
    "label": "Replacement cost as % of annual salary",
}])
scenario_inputs.to_csv(OUTPUTS_DIR / "scenario_inputs.csv", index=False)
print(f"\nSaved: {OUTPUTS_DIR / 'scenario_inputs.csv'}")


### Step 4.3 - Three scenario definitions for Excel Scenario Manager

These are the three named scenarios that will appear in Excel's Scenario Manager. We anchor them to the actual India macro range (2015-2025) so the executive can interpret each as a plausible state of the world.

In [ ]:
scenarios = [
    {
        "scenario_name": "Best Case (low CPI, low rates)",
        "cpi_pct":            3.30,    # 2025 actual
        "repo_rate_pct":      5.25,    # 2025 actual
        "salary_inflation_pct": 6.50,  # CPI + premium for talent
        "attrition_multiplier": 0.85,  # macro stability tempers attrition
        "rationale": "Anchored to India 2025 - inflation moderating, rate-cut cycle. "
                      "Low cost of capital, talent market less heated.",
    },
    {
        "scenario_name": "Base Case (median 2015-25)",
        "cpi_pct":            5.00,
        "repo_rate_pct":      6.25,
        "salary_inflation_pct": 8.00,
        "attrition_multiplier": 1.00,
        "rationale": "Anchored to India median over 2015-2025. The most likely steady-state.",
    },
    {
        "scenario_name": "Stress Case (high CPI, tight policy)",
        "cpi_pct":            6.70,    # 2022 actual (post-COVID inflation peak)
        "repo_rate_pct":      6.50,    # 2022/23 actual peak
        "salary_inflation_pct": 11.00, # talent demands real wage protection
        "attrition_multiplier": 1.25,  # poaching intensifies in tight markets
        "rationale": "Anchored to India 2022/23 - peak post-COVID inflation, RBI tightening. "
                      "Talent attrition rises as competitors offer inflation-protection raises.",
    },
]
scenarios_df = pd.DataFrame(scenarios)
print("Three scenarios for the Excel model:")
print(scenarios_df[["scenario_name","cpi_pct","repo_rate_pct","salary_inflation_pct","attrition_multiplier"]].to_string(index=False))

# Pre-compute outcomes for each scenario (the model's Output cells)
results = []
ann_inc_rd    = baseline_avg_income_rd * 12
ann_inc_sales = baseline_avg_income_sales * 12
ann_inc_hr    = baseline_avg_income_hr * 12

for s in scenarios:
    sal_factor = 1 + s["salary_inflation_pct"]/100  # year-1 salary uplift
    attr_factor = s["attrition_multiplier"]

    hc_rd    = round(PLAN_HEADCOUNT * PLAN_RD_SHARE)
    hc_sales = round(PLAN_HEADCOUNT * 0.25)
    hc_hr    = PLAN_HEADCOUNT - hc_rd - hc_sales

    # Annual salary cost
    cost_rd    = hc_rd    * ann_inc_rd    * sal_factor
    cost_sales = hc_sales * ann_inc_sales * sal_factor
    cost_hr    = hc_hr    * ann_inc_hr    * sal_factor
    total_salary_cost = cost_rd + cost_sales + cost_hr

    # Expected attrition cost
    exp_leavers_rd    = hc_rd    * baseline_attrition_rd    * attr_factor
    exp_leavers_sales = hc_sales * baseline_attrition_sales * attr_factor
    exp_leavers_hr    = hc_hr    * baseline_attrition_hr    * attr_factor
    total_leavers     = exp_leavers_rd + exp_leavers_sales + exp_leavers_hr

    attrition_cost_rd    = exp_leavers_rd    * ann_inc_rd    * sal_factor * COST_OF_REPLACEMENT_PCT
    attrition_cost_sales = exp_leavers_sales * ann_inc_sales * sal_factor * COST_OF_REPLACEMENT_PCT
    attrition_cost_hr    = exp_leavers_hr    * ann_inc_hr    * sal_factor * COST_OF_REPLACEMENT_PCT
    total_attrition_cost = attrition_cost_rd + attrition_cost_sales + attrition_cost_hr

    # Total operating cost
    total_year1 = total_salary_cost + total_attrition_cost

    # NPV proxy: discount at repo rate
    discount = s["repo_rate_pct"] / 100
    real_cost_year1 = total_year1 / (1 + discount)

    results.append({
        "scenario":               s["scenario_name"],
        "headcount_total":        PLAN_HEADCOUNT,
        "expected_leavers":       round(total_leavers, 1),
        "salary_cost_year1":      round(total_salary_cost),
        "attrition_cost_year1":   round(total_attrition_cost),
        "total_year1_cost":       round(total_year1),
        "real_cost_at_repo_rate": round(real_cost_year1),
    })

scenarios_results_df = pd.DataFrame(results)
print("\nPre-computed Year-1 cost outcomes (for verification of the Excel model):")
print(scenarios_results_df.to_string(index=False))

scenarios_results_df.to_csv(OUTPUTS_DIR / "scenario_outcomes.csv", index=False)
scenarios_df.to_csv(OUTPUTS_DIR / "scenario_definitions.csv", index=False)


## Part 5 - Final summary chart

The single chart that goes on the cover of the executive deck.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Top-left: Tenure cohort attrition (the "rolling" chart)
ax = axes[0, 0]
bars = ax.bar(cohort_attrition["TenureBucket"].astype(str), cohort_attrition["attrition_rate_pct"],
                color=["#C00000","#E07A1F","#E0C200","#7DB45F","#2E7D32"],
                edgecolor="black", linewidth=0.8)
for b, p in zip(bars, cohort_attrition["attrition_rate_pct"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
            f"{p:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax.set_title("Attrition by Tenure Cohort\n(rolling-attrition proxy)", fontweight="bold")
ax.set_xlabel("Years at company")
ax.set_ylabel("Attrition rate (%)")
ax.set_ylim(0, 40)

# Top-right: Department attrition
ax = axes[0, 1]
dept_sorted = dept_attrition.sort_values("attrition_rate_pct", ascending=True)
bars = ax.barh(dept_sorted["Department"], dept_sorted["attrition_rate_pct"],
                color="#1F3864", edgecolor="black", linewidth=0.8)
for b, p in zip(bars, dept_sorted["attrition_rate_pct"]):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2,
            f"{p:.1f}%", va="center", fontsize=10, fontweight="bold")
ax.set_title("Attrition by Department", fontweight="bold")
ax.set_xlabel("Attrition rate (%)")
ax.set_xlim(0, 25)

# Bottom-left: CIs for morale
ax = axes[1, 0]
y_pos = range(len(morale_df))
ax.errorbar(morale_df["sample_mean"], y_pos,
             xerr=[morale_df["sample_mean"]-morale_df["ci_95_low"],
                   morale_df["ci_95_high"]-morale_df["sample_mean"]],
             fmt="o", markersize=12, color="#1F3864", linewidth=2.5, capsize=10)
ax.scatter(morale_df["population_mean"], y_pos, marker="x", s=150, color="#C00000",
            linewidth=2.5, label="True population mean", zorder=5)
ax.set_yticks(y_pos)
ax.set_yticklabels([d.replace("Satisfaction","Sat.").replace("Involvement","Involve.")
                    for d in morale_df["dimension"]])
ax.set_xlim(2.0, 3.5)
ax.axvline(2.5, color="grey", linestyle=":", alpha=0.5)
ax.set_title("Morale 95% Confidence Intervals\n(from sample n=100, vs true population)",
             fontweight="bold")
ax.set_xlabel("Mean score (4-point Likert)")
ax.legend(loc="lower right")

# Bottom-right: scenario costs
ax = axes[1, 1]
scen_short = ["Best", "Base", "Stress"]
salary_costs = scenarios_results_df["salary_cost_year1"] / 1e6
attrition_costs = scenarios_results_df["attrition_cost_year1"] / 1e6
ax.bar(scen_short, salary_costs, color="#1F3864", label="Salary cost", edgecolor="black")
ax.bar(scen_short, attrition_costs, bottom=salary_costs, color="#C00000",
        label="Attrition cost", edgecolor="black")
for i, (s, a) in enumerate(zip(salary_costs, attrition_costs)):
    ax.text(i, s+a + 0.5, f"{s+a:.1f}M", ha="center", fontsize=11, fontweight="bold")
ax.set_title("Year-1 Total Cost by Scenario\n(in millions of monthly-income units)",
             fontweight="bold")
ax.set_ylabel("Cost (millions)")
ax.legend(loc="upper left")

plt.suptitle("Consult-Tech Executive Summary - Workforce Stability & Market Equilibrium",
             fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "executive_summary.png", dpi=160, bbox_inches="tight")
plt.show()

print("\n=== ANALYTICS NOTEBOOK COMPLETE ===")
print(f"Outputs in {OUTPUTS_DIR}:")
print("  hr_cleaned.csv              - cleaned data for Power BI")
print("  cohort_attrition.csv        - rolling attrition rate")
print("  department_attrition.csv    - dept drill-down")
print("  role_attrition.csv          - role drill-down")
print("  market_structure_attrition.csv - BusinessTravel proxy")
print("  clt_simulation_results.csv  - CLT validation table")
print("  confidence_intervals.csv    - 90/95/99% CIs")
print("  morale_snapshot.csv         - all 5 satisfaction CIs")
print("  scenario_inputs.csv         - inputs for Excel model")
print("  scenario_definitions.csv    - 3 scenario definitions")
print("  scenario_outcomes.csv       - pre-computed cost outcomes")
print(f"\nFigures in {FIGURES_DIR}:")
print("  clt_sampling_distributions.png")
print("  confidence_intervals_morale.png")
print("  executive_summary.png")


**Notebook complete.** The other two deliverables (Excel `.xlsx` and Power BI `.pbix` build spec) consume the CSVs produced here. Build them next:

1. Run the Excel builder script to produce `Market_Feasibility_Model.xlsx` with Scenario Manager pre-configured
2. Read `PowerBI_Build_Spec.md` to assemble the dashboard - every Power Query M snippet, DAX measure, and visual layout is documented there